In [1]:
import pickle

import pandas as pd
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error
import mlflow 

/home/yagoub/Desktop/MLOps-lab/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from pathlib import Path

mlflow_db = (Path("../../../mlflow.db")).resolve()
mlflow.set_tracking_uri(f"sqlite:///{mlflow_db}")
mlflow.set_experiment("nyc-taxi-exp1")

<Experiment: artifact_location='/home/yagoub/Desktop/MLOps-lab/experiments/nyc-taxi-duration/notebooks/mlruns/1', creation_time=1781826743726, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1781826743726, lifecycle_stage='active', name='nyc-taxi-exp1', tags={}, trace_location=None, workspace='default'>

In [3]:
def read_dataframe(filename):
    if filename.endswith('.csv'):
        df = pd.read_csv(filename)

        df.lpep_dropoff_datetime = pd.to_datetime(df.lpep_dropoff_datetime)
        df.lpep_pickup_datetime = pd.to_datetime(df.lpep_pickup_datetime)
    elif filename.endswith('.parquet'):
        df = pd.read_parquet(filename)

    df['duration'] = df.tpep_dropoff_datetime - df.tpep_pickup_datetime
    df.duration = df.duration.dt.total_seconds() / 60

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    
    return df

In [16]:
TRAIN_SAMPLE_SIZE = 15_000
VAL_SAMPLE_SIZE = 5_000

df_train = read_dataframe('../../../data/raw/yellow_tripdata_2023-01.parquet')
df_train = df_train.sample(n=min(TRAIN_SAMPLE_SIZE, len(df_train)), random_state=42)

In [17]:
df_train.shape

(15000, 20)

In [18]:
df_val = read_dataframe('../../../data/raw/yellow_tripdata_2023-02.parquet')
df_val = df_val.sample(n=min(VAL_SAMPLE_SIZE, len(df_val)), random_state=42)

In [19]:
df_train['PU_DO'] = df_train['PULocationID'] + '_' + df_train['DOLocationID']
df_val['PU_DO'] = df_val['PULocationID'] + '_' + df_val['DOLocationID']

In [20]:
categorical = ['PU_DO'] #'PULocationID', 'DOLocationID']
numerical = ['trip_distance']

dv = DictVectorizer()

train_dicts = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

In [21]:
val_dicts = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dicts)

In [22]:
target = 'duration'
y_train = df_train[target].values
y_val = df_val[target].values

In [11]:
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_val)

root_mean_squared_error(y_val, y_pred)

5.715634233168297

In [13]:
with open('../../../models/lin_reg.bin', 'wb') as f_out:
    pickle.dump((dv, lr), f_out)

In [23]:
import gc

import xgboost as xgb
from hyperopt import STATUS_OK, Trials, fmin, hp, tpe
from hyperopt.pyll import scope

In [24]:
train = xgb.DMatrix(X_train, label=y_train)
valid = xgb.DMatrix(X_val, label=y_val)

In [25]:
def objective(params):
    with mlflow.start_run():
        mlflow.set_tag("model", "xgboost")
        mlflow.log_params(params)

        booster = xgb.train(
            params=params,
            dtrain=train,
            num_boost_round=100,
            evals=[(valid, "validation")],
            early_stopping_rounds=10,
            verbose_eval=False,
        )

        y_pred = booster.predict(valid)
        rmse = root_mean_squared_error(y_val, y_pred)

        mlflow.log_metric("rmse", rmse)
        mlflow.log_metric("best_iteration", booster.best_iteration)

        del booster
        del y_pred
        gc.collect()

    return {"loss": rmse, "status": STATUS_OK}

In [26]:
search_space = {
    "max_depth": scope.int(hp.quniform("max_depth", 3, 6, 1)),
    "learning_rate": hp.loguniform("learning_rate", -2.5, -1),
    "reg_alpha": hp.loguniform("reg_alpha", -3, -1),
    "reg_lambda": hp.loguniform("reg_lambda", -3, -1),
    "min_child_weight": hp.quniform("min_child_weight", 1, 5, 1),

    "objective": "reg:squarederror",
    "eval_metric": "rmse",

    "tree_method": "hist",
    "max_bin": 64,
    "nthread": 4,
    "seed": 42,
    "verbosity": 0,
}

best_result = fmin(
    fn=objective,
    space=search_space,
    algo=tpe.suggest,
    max_evals=10,
    trials=Trials(),
)

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [00:04<00:00,  2.24trial/s, best loss: 5.0510867323984225]
